In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OCQ_F.xpt"
csv_file = DATA_DIR / "OCQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 OCQ_F XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OCD150",
    "OCQ180",
    "OCQ210",
    "OCD231",
    "OCD241",
    "OCQ260",
    "OCQ265",
    "OCD270",
    "OCQ290G",
    "OCQ290Q",
    "OCQ380",
    "OCD390G",
    "OCD391",
    "OCD392",
    "OCD395",
    "OCQ510",
    "OCQ520",
    "OCQ530",
    "OCQ540",
    "OCQ550",
    "OCQ560",
    "OCQ570",
    "OCQ580"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "OCQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 6889:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 24:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,889 rows x 24 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally replace this value.
#
# First confirm:
#
#     variable
#     count
#     CDC meaning/range
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated against:
#
#     attached OCQ_F.xpt
#     CDC 2009-2010 OCQ_F codebook
#
#
# OCD270:
#     0 = Less than 1 month
#
# OCQ290Q:
#     0 = exposure occurred < 1 hour/day
#
# OCD395:
#     0 = Less than 1 month
#
# OCQ520 / OCQ540 / OCQ560 / OCQ580:
#     interviewer instruction:
#     IF LESS THAN 1 YEAR, ENTER 0
#

EXPECTED_TINY_COUNTS = {

    "OCD270": 77,

    "OCQ290Q": 26,

    "OCD395": 18,

    "OCQ520": 173,

    "OCQ540": 115,

    "OCQ560": 76,

    "OCQ580": 163
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated OCQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 648:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Exact 648-value tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 648:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# dataset SHOULD contain.
#
# Only validated:
#
#     TINY_VALUE -> 0
#
# positions are changed.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# This test MUST occur BEFORE Int64 conversion.
#
# For OCQ_F:
#
#     genuine decimal variables = NONE
#
# If fractions unexpectedly appear, STOP.
#
# This same safeguard protects genuine Dietary,
# Lab, Examination, and weight decimals.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in OCQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# After zero restoration and the fractional guard,
# all 24 fields are proven whole-number fields.
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimals "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All OCQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CATEGORICAL FREQUENCY CHECKS
# ============================================================

#
# Major categorical variables from the CDC codebook.
#

CDC_CATEGORICAL = {

    "OCD150": {
        "values": {
            1: 3371,
            2: 132,
            3: 279,
            4: 3106,
            9: 1
        },
        "missing": 0
    },

    "OCQ210": {
        "values": {
            1: 365,
            2: 732,
            7: 1,
            9: 2
        },
        "missing": 5789
    },

    "OCD231": {
        "values": {
            1: 82,
            2: 26,
            3: 22,
            4: 242,
            5: 248,
            6: 162,
            7: 97,
            8: 359,
            9: 157,
            10: 72,
            11: 105,
            12: 66,
            13: 164,
            14: 180,
            15: 299,
            16: 455,
            17: 74,
            18: 335,
            19: 165,
            20: 65,
            21: 123,
            98: 5
        },
        "missing": 3386
    },

    "OCD241": {
        "values": {
            1: 337,
            2: 103,
            3: 61,
            4: 43,
            5: 41,
            6: 64,
            7: 26,
            8: 174,
            9: 66,
            10: 146,
            11: 109,
            12: 70,
            13: 264,
            14: 271,
            15: 122,
            16: 350,
            17: 379,
            18: 38,
            19: 210,
            20: 95,
            21: 276,
            22: 252,
            98: 6
        },
        "missing": 3386
    },

    "OCQ260": {
        "values": {
            1: 2584,
            2: 74,
            3: 223,
            4: 209,
            5: 388,
            6: 11,
            77: 5,
            99: 9
        },
        "missing": 3386
    },

    "OCQ265": {
        "values": {
            1: 2529,
            2: 209,
            3: 180,
            4: 320,
            5: 262,
            7: 1,
            9: 2
        },
        "missing": 3386
    },

    "OCQ290G": {
        "values": {
            1: 482,
            2: 3015,
            7: 1,
            9: 5
        },
        "missing": 3386
    },

    "OCQ380": {
        "values": {
            1: 473,
            2: 444,
            3: 1227,
            4: 233,
            5: 147,
            6: 358,
            7: 220
        },
        "missing": 3787
    },

    "OCD390G": {
        "values": {
            1: 3749,
            2: 2559,
            3: 63,
            4: 508,
            7: 5,
            9: 5
        },
        "missing": 0
    },

    "OCQ510": {
        "values": {
            1: 1854,
            2: 4110,
            9: 16
        },
        "missing": 909
    },

    "OCQ530": {
        "values": {
            1: 1330,
            2: 4644,
            9: 6
        },
        "missing": 909
    },

    "OCQ550": {
        "values": {
            1: 1410,
            2: 4566,
            9: 4
        },
        "missing": 909
    },

    "OCQ570": {
        "values": {
            1: 1897,
            2: 4078,
            9: 5
        },
        "missing": 909
    }
}


print("\n--- CDC CATEGORICAL CHECKS ---")


for col, expected in CDC_CATEGORICAL.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - OCQ180
# ============================================================

#
# Hours worked last week at all jobs
#
# CDC:
#
# 1-130  = 3,367
# 77777  = 1
# 99999  = 3
# Missing = 3,518
#

s = df["OCQ180"]


if not (
    int(
        s.between(
            1,
            130
        ).sum()
    )
    == 3367
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 1
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 3
    and
    int(
        s.isna()
        .sum()
    )
    == 3518
):

    raise ValueError(
        "OCQ180 differs from CDC."
    )


print(
    "PASS: OCQ180 matches CDC."
)


# ============================================================
# 13. CDC CHECK - OCD270
# ============================================================

#
# Number months working in main job
#
# CDC:
#
# 1-720 = 3,419
# 0     = Less than 1 month = 77
# 77777 = 3
# 99999 = 4
# Missing = 3,386
#
# The 77 raw tiny values therefore represent zero.
#

s = df["OCD270"]


print("\n--- OCD270 ---")

print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            720
        ).sum()
    )
    == 3419
    and
    int(
        s.eq(0)
        .sum()
    )
    == 77
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 3
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 4
    and
    int(
        s.isna()
        .sum()
    )
    == 3386
):

    raise ValueError(
        "OCD270 differs from CDC / zero restoration."
    )


print(
    "PASS: OCD270 matches CDC."
)


# ============================================================
# 14. CDC CHECK - OCQ290Q
# ============================================================

#
# Hours/day smell tobacco smoke at work
#
# CDC:
#
# 0-24 = 482
# Missing = 6,407
#
# CDC explicitly says:
#
# IF NUMBER OF HOURS IS LESS THAN ONE, ENTER ZERO.
#
# Attached XPT:
#
# zero = 26
#

s = df["OCQ290Q"]


print("\n--- OCQ290Q ---")

print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            24
        ).sum()
    )
    == 482
    and
    int(
        s.eq(0)
        .sum()
    )
    == 26
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 6407
):

    raise ValueError(
        "OCQ290Q differs from CDC / zero restoration."
    )


print(
    "PASS: OCQ290Q matches CDC."
)


# ============================================================
# 15. CDC CHECK - OCD395
# ============================================================

#
# Duration of longest job in months
#
# CDC:
#
# 1-720 = 3,791
# 0     = Less than 1 month = 18
# 77777 = 1
# 99999 = 2
# Missing = 3,077
#

s = df["OCD395"]


print("\n--- OCD395 ---")

print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            720
        ).sum()
    )
    == 3791
    and
    int(
        s.eq(0)
        .sum()
    )
    == 18
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 1
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 2
    and
    int(
        s.isna()
        .sum()
    )
    == 3077
):

    raise ValueError(
        "OCD395 differs from CDC / zero restoration."
    )


print(
    "PASS: OCD395 matches CDC."
)


# ============================================================
# 16. CDC CHECK - EXPOSURE YEAR VARIABLES
# ============================================================

#
# CDC instructions for each:
#
# IF LESS THAN 1 YEAR, ENTER 0.
#
#
# OCQ520:
# mineral dust
#
#     0-60  = 1,849
#     9999  = 5
#     Missing = 5,035
#     Zero = 173
#
#
# OCQ540:
# organic dust
#
#     0-78  = 1,325
#     9999  = 5
#     Missing = 5,559
#     Zero = 115
#
#
# OCQ560:
# exhaust fumes
#
#     0-60  = 1,407
#     9999  = 3
#     Missing = 5,479
#     Zero = 76
#
#
# OCQ580:
# other fumes
#
#     0-60  = 1,894
#     9999  = 3
#     Missing = 4,992
#     Zero = 163
#

EXPOSURE_CHECKS = {

    "OCQ520": {
        "min": 0,
        "max": 60,
        "range_count": 1849,
        "zero": 173,
        "refused": 0,
        "dontknow": 5,
        "missing": 5035
    },

    "OCQ540": {
        "min": 0,
        "max": 78,
        "range_count": 1325,
        "zero": 115,
        "refused": 0,
        "dontknow": 5,
        "missing": 5559
    },

    "OCQ560": {
        "min": 0,
        "max": 60,
        "range_count": 1407,
        "zero": 76,
        "refused": 0,
        "dontknow": 3,
        "missing": 5479
    },

    "OCQ580": {
        "min": 0,
        "max": 60,
        "range_count": 1894,
        "zero": 163,
        "refused": 0,
        "dontknow": 3,
        "missing": 4992
    }
}


print("\n--- CDC EXPOSURE-YEAR CHECKS ---")


for col, expected in EXPOSURE_CHECKS.items():

    s = df[col]


    range_count = int(
        s.between(
            expected["min"],
            expected["max"]
        ).sum()
    )


    zero_count = int(
        s.eq(0)
        .sum()
    )


    refused_count = int(
        s.eq(7777)
        .sum()
    )


    dontknow_count = int(
        s.eq(9999)
        .sum()
    )


    missing_count = int(
        s.isna()
        .sum()
    )


    if range_count != expected["range_count"]:

        raise ValueError(
            f"{col}: CDC range count differs."
        )


    if zero_count != expected["zero"]:

        raise ValueError(
            f"{col}: zero count differs."
        )


    if refused_count != expected["refused"]:

        raise ValueError(
            f"{col}: refused count differs."
        )


    if dontknow_count != expected["dontknow"]:

        raise ValueError(
            f"{col}: don't-know count differs."
        )


    if missing_count != expected["missing"]:

        raise ValueError(
            f"{col}: missing count differs."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 17. VALIDATED ZERO COUNTS
# ============================================================

ZERO_COUNTS = {
    "OCD270": 77,
    "OCQ290Q": 26,
    "OCD395": 18,
    "OCQ520": 173,
    "OCQ540": 115,
    "OCQ560": 76,
    "OCQ580": 163
}


print("\n--- ZERO RESTORATION CHECK ---")


for col, expected_zero_count in ZERO_COUNTS.items():

    actual_zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col}: {actual_zero_count} zeros"
    )


    if actual_zero_count != expected_zero_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_zero_count} zeros, "
            f"found {actual_zero_count}."
        )


print(
    "PASS: All validated zero counts are correct."
)


# ============================================================
# 18. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected changes from raw pandas XPT:
#
# OCD270   tiny -> 0     77
# OCQ290Q  tiny -> 0     26
# OCD395   tiny -> 0     18
# OCQ520   tiny -> 0    173
# OCQ540   tiny -> 0    115
# OCQ560   tiny -> 0     76
# OCQ580   tiny -> 0    163
#
# TOTAL = 648
#
# NOTHING ELSE may change numerically.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 24 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# All OCQ_F variables are proven whole-number fields
# after validated zero restoration.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 20. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 21. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 24 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 22. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 23. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 24. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# OCQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not occur anywhere in this
# particular final CSV.
#
# IMPORTANT:
#
# Do NOT use this global test on Dietary or other
# datasets containing real decimals.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 25. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 26. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "OCQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "OCD270 zero restorations:",
    77
)


print(
    "OCQ290Q zero restorations:",
    26
)


print(
    "OCD395 zero restorations:",
    18
)


print(
    "OCQ520 zero restorations:",
    173
)


print(
    "OCQ540 zero restorations:",
    115
)


print(
    "OCQ560 zero restorations:",
    76
)


print(
    "OCQ580 zero restorations:",
    163
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 OCQ_F XPT -> CSV
Rows: 6,889
Columns: 24
PASS: Structure = 6,889 rows x 24 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 648

Tiny values by variable:
OCQ520     173
OCQ580     163
OCQ540     115
OCD270      77
OCQ560      76
OCQ290Q     26
OCD395      18

PASS: Exact 648-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
OCD270: 77 restored to 0
OCQ290Q: 26 restored to 0
OCD395: 18 restored to 0
OCQ520: 173 restored to 0
OCQ540: 115 restored to 0
OCQ560: 76 restored to 0
OCQ580: 163 restored to 0

PASS: 648 validated zeros restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in OCQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All OCQ_F fields formatted without unnecessary .0.

--- CDC CATEGORICAL CHECKS ---
PASS: OCD150 matches CDC.
PASS: OCQ210 matches CDC.
PASS: OCD231 matches CDC.
PASS: OCD241 matches CDC.
PASS: OCQ260 matches